### Mugrade boilerplate

In [ ]:
### Run this cell to install and import the homework tests
!pip install --upgrade git+https://github.com/locuslab/mugrade.git
!wget -nc https://raw.githubusercontent.com/zkolter/llm_speedrun/refs/heads/main/part3_llm_training_tests.py

import mugrade
import os
from part3_llm_training_tests import *

def _mugrade_name(name):
    def rename(function):
        function.__name__ = name
        return function
    return rename

os.environ["MUGRADE_HW"] = "Part 3 - LLM Training"
os.environ["MUGRADE_KEY"] = "" ### Your key here

### BPE

Insert the BPE tokenizer you built in Part 1.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE

### LLM Architecture

Insert your LLM architecture from Part 2.

In [ ]:
### BEGIN YOUR CODE
pass
### END YOUR CODE

### Downloading necessary files

These next lines will donwload the necessary tokenizer and pre-tokenized data files, which you can build in Part 1 (but which require a fairly substantial machine to run as-is).

In [ ]:
from huggingface_hub import hf_hub_download
import os

repo = "zkolter/llm_speedrun"
filenames = ["fineweb-edu-10BT.shuffle.bin", "smoltalk.shuffle.bin", "tokenizer_50M.bpe"]

for filename in filenames:
    if not os.path.exists(filename):
        hf_hub_download(repo_id=repo, filename=filename, repo_type="dataset", local_dir=".")

You can use the following config file for training.  This is intended to load a batch size that will fit comfortably on a GPU with 80GB of memory, you can adjust as needed.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 16,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 7e-4,
    "weight_decay": 0.025,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 1
}

### LLM Training

In [ ]:
import wandb
import time
from array import array
import os


# @mugrade.local_tests
def cross_entropy_loss(logits, y):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

class Adam:
    # @mugrade.local_tests
    @_mugrade_name("Adam_init")
    def __init__(self, params, lr_schedule, betas = (0.9, 0.95), eps=1e-5, weight_decay=0.0):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def step(self):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

class LRSchedule:
    # @mugrade.local_tests
    @_mugrade_name("LRSchedule_init")
    def __init__(self, total_steps, lr=1e-3, warmup_steps=50, decay_ratio=0.4, min_frac=0.1):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE

    # @mugrade.local_tests
    def get_lr(self, step):
        ### BEGIN YOUR CODE
        pass
        ### END YOUR CODE


# @mugrade.local_tests
def train_llm(config, log=False):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

If your code passes all the tests, you can (optionally) uncomment this block to train the d12 model, here done on a single GPU.

In [ ]:
with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]

llm = train_llm(config, log=True)

### Distributed LLM Training

Implement a distributed version of the training code, starting from the version above.  Note that there was one error in the class version, that `config["batch_size"]` should be replaced by `local_batch_size` in the line that computes bpb for logging.

In [ ]:
# @mugrade.local_tests
def train_llm_distributed(rank, nccl_uid, config, log=False):
    ### BEGIN YOUR CODE
    pass
    ### END YOUR CODE

If you're able to access a machine with 8 GPUs, then the following code will launch the distributed training run with a different config that uses a larger batch size.

In [ ]:
%%writefile config.d12.json
{
    "depth": 12,
    "aspect_ratio": 64,
    "mlp_multiple": 4,
    "head_dim": 128,
    "dtype": "bfloat16",
    "vocab_size": 32768,
    "batch_size": 128,
    "seq_len": 2048,
    "rope_theta": 10000,
    "tokenizer": "tokenizer_50M.bpe",
    "token_multiple": 20,
    "lr": 1e-3,
    "weight_decay": 0.1,
    "data_mix": {
        "fineweb-edu-10BT.shuffle.bin": 0.875,
        "smoltalk.shuffle.bin": 0.125
    },
    "num_gpus": 8
}

In [ ]:
from joblib import Parallel, delayed
os.environ["NCCL_NVLS_ENABLE"] = "0"  # shouldn't be needed if your system isn't messed up like mine

with open("config.d12.class.json", "rt") as f:
    config = json.load(f)
torch_types = {"float32": torch.float32, "bfloat16": torch.bfloat16}
config["dtype"] = torch_types[config["dtype"]]
nccl_uid = torch.cuda.nccl.unique_id()

Parallel(n_jobs = config["num_gpus"], backend="loky")(
    delayed(train_llm_distributed)(i, nccl_uid, config, log=True) for i in range(config["num_gpus"])
)
